# Recess Ready: Philadelphia

**Question:** each evening, can we predict whether tomorrow will be unsafe for outdoor recess at Philadelphia schools, and **why** (heat, cold, bad air, ozone smog, flash-flood rain, dangerous wind, thunderstorms, heavy snow, or ice)?

*Prototype, not official guidance. Always follow official National Weather Service alerts and your school district.*

## Step 1: Download Philadelphia's daily weather and air quality

We ask Open-Meteo (free, no API key) for one row per day:
- **Weather** (archive API): hottest and coldest "feels like" temperature, rain, snow, strongest wind gust, **wind speed and direction**, and a **weather code**. The weather code is a number that says the worst weather of the day, for example 95 = thunderstorm and 66 = freezing rain.
- **Air quality** (air-quality API), every hour, turned into one number per day:
  - **PM2.5** and **PM10**: tiny and small particles (smoke, dust, exhaust)
  - **Ozone**: smog that forms on hot, sunny days
  - **Nitrogen dioxide (NO₂)**: mostly from cars, trucks and power plants
  - **US AQI**: the official EPA Air Quality Index (0-500) that combines all of these. For Philadelphia we also get each pollutant's own AQI, so we can say which one is the **main pollutant** each day.

We write this as two small **functions** (`download` and `daily_table`) because later we reuse them for 35 places around Philadelphia, and we want every place to be handled exactly the same way.

Note: Open-Meteo's air quality numbers are **estimates from a computer model (CAMS)**, not readings from a sensor. We say this in our write-up.

**Saved data:** the first run downloads everything and saves it as CSV files in `data/`. Later runs read those files instead, which is faster and works even if Open-Meteo is slow or down. To download fresh data, delete the files in `data/`.

In [ ]:
import pandas as pd
import requests
import time
from pathlib import Path

# Downloaded data is saved here, so later runs (and judges) don't need to download it again
DATA_DIR = Path("data") if Path("data").exists() else Path("../data")
DATA_DIR.mkdir(exist_ok=True)

LAT, LON = 39.95, -75.16            # Philadelphia (Center City)
START, END = "2022-08-01", "2026-11-12"

WEATHER_URL = "https://archive-api.open-meteo.com/v1/archive"
AIR_URL = "https://air-quality-api.open-meteo.com/v1/air-quality"
AIR_VARS = "pm2_5,ozone,pm10,nitrogen_dioxide,us_aqi"
AQI_PARTS = ("us_aqi_pm2_5,us_aqi_pm10,us_aqi_nitrogen_dioxide,us_aqi_ozone,"
             "us_aqi_sulphur_dioxide,us_aqi_carbon_monoxide")   # each pollutant's own AQI
DAILY_VARS = ("apparent_temperature_max,apparent_temperature_min,precipitation_sum,snowfall_sum,"
              "wind_gusts_10m_max,weather_code,wind_speed_10m_max,wind_direction_10m_dominant")

def get_json(url, params, tries=4):
    # Ask a website for data; if it's slow or busy, wait a bit and try again
    for attempt in range(tries):
        try:
            r = requests.get(url, params=params, timeout=120)
            r.raise_for_status()
            return r.json()
        except Exception as err:
            if attempt == tries - 1:
                raise
            print(f"Download problem ({type(err).__name__}), trying again in {10 * (attempt + 1)} seconds...")
            time.sleep(10 * (attempt + 1))

def download(lats, lons, start, end, tz="America/New_York", air_vars=AIR_VARS, batch=5):
    # Ask Open-Meteo for many places, a few at a time (big requests can time out)
    weather, air = [], []
    for i in range(0, len(lats), batch):
        where = {"latitude": ",".join(str(x) for x in lats[i:i + batch]),
                 "longitude": ",".join(str(x) for x in lons[i:i + batch]),
                 "start_date": start, "end_date": end, "timezone": tz}
        w = get_json(WEATHER_URL, {**where, "daily": DAILY_VARS})
        a = get_json(AIR_URL, {**where, "hourly": air_vars})
        if isinstance(w, dict):       # one place comes back as one answer; make it a list
            w, a = [w], [a]
        weather += w
        air += a
    return weather, air

def cached(name, make_table):
    # Use the saved CSV in data/ if we have one; otherwise download, then save it
    path = DATA_DIR / f"{name}.csv"
    if path.exists():
        print(f"Using saved data: {path}")
        return pd.read_csv(path, parse_dates=["time"])
    table = make_table()
    table.to_csv(path, index=False)
    print(f"Downloaded and saved: {path}")
    return table

def daily_table(weather, air):
    # Turn one place's answers into a table with one row per day
    w = pd.DataFrame(weather["daily"])
    w["time"] = pd.to_datetime(w["time"])
    a = pd.DataFrame(air["hourly"]).set_index("time")
    a.index = pd.to_datetime(a.index)
    air_daily = pd.DataFrame({
        "pm2_5": a["pm2_5"].resample("D").mean(),                     # average of the 24 hours
        "ozone_8h": a["ozone"].rolling(8).mean().resample("D").max(),  # worst 8-hour average (EPA style)
        "pm10": a["pm10"].resample("D").mean(),
        "nitrogen_dioxide": a["nitrogen_dioxide"].resample("D").mean(),
        "us_aqi": a["us_aqi"].resample("D").max(),                     # the day's worst AQI
    })
    parts = [c for c in a.columns if c.startswith("us_aqi_")]
    if parts:   # which pollutant had the highest AQI that day?
        worst = a[parts].resample("D").max().dropna(how="all")   # skip days with no AQI data at all
        air_daily["main_pollutant"] = worst.idxmax(axis=1).str.replace("us_aqi_", "")
    return w.merge(air_daily.reset_index(), on="time")

def philly_table():
    weather, air = download([LAT], [LON], START, END, air_vars=AIR_VARS + "," + AQI_PARTS)
    return daily_table(weather[0], air[0])

df = cached("philly_daily", philly_table)
print(df.shape)
df.head()

## Step 2: How many unsafe *school* days are there, and why?

A day is **unsafe** if any one of these happens. ⚠️ **Team to-do:** look up and cite a real source for each threshold (e.g. NWS, US EPA, or our school district's recess policy), and change the number if the source says something different.

| Warning | Rule |
|---|---|
| Too hot | "feels like" max ≥ 32 °C (≈90 °F, NWS "extreme caution") |
| Too cold | "feels like" min ≤ −7 °C (≈20 °F) |
| Bad air (smoke/particles) | daily PM2.5 > 35.4 µg/m³ (US EPA "Unhealthy for Sensitive Groups") |
| Ozone smog | worst 8-hour ozone > 137 µg/m³ (≈70 ppb, US EPA standard) |
| Flash flood / heavy rain | ≥ 25 mm (≈1 inch) of rain in the day |
| Dangerous wind (hurricanes, storms) | gusts ≥ 74 km/h (≈46 mph, NWS wind advisory level) |
| Thunderstorm / lightning | weather code 95, 96 or 99 |
| Heavy snow | ≥ 5 cm (≈2 inches) of snow |
| Ice / freezing rain | weather code 56, 57, 66 or 67 |

**Left out on purpose:** tornadoes (not in our data, very rare in Philadelphia; "dangerous wind" and "thunderstorm" cover the storms that bring them), earthquakes (nobody can predict them, and they aren't climate), and UV/sunburn (not in our weather data).

We only count **school days**: weekdays from September to June (no recess in July and August). We ignore holidays for now.

**Why this check comes first:** if a warning almost never happens, the model can't learn it. We'd drop it or change the threshold now, not on Sunday.

In [ ]:
hazards = ["too_hot", "too_cold", "bad_air", "ozone_smog", "heavy_rain",
           "high_wind", "thunderstorm", "heavy_snow", "ice"]

def add_hazards(t):
    # Mark each day True/False for each danger, using the rules in the table above
    t["too_hot"] = t["apparent_temperature_max"] >= 32
    t["too_cold"] = t["apparent_temperature_min"] <= -7
    t["bad_air"] = t["pm2_5"] > 35.4
    t["ozone_smog"] = t["ozone_8h"] > 137
    t["heavy_rain"] = t["precipitation_sum"] >= 25
    t["high_wind"] = t["wind_gusts_10m_max"] >= 74
    t["thunderstorm"] = t["weather_code"].isin([95, 96, 99])
    t["heavy_snow"] = t["snowfall_sum"] >= 5
    t["ice"] = t["weather_code"].isin([56, 57, 66, 67])
    t["unsafe"] = t[hazards].any(axis=1)   # True if ANY danger happens
    return t

def is_school_day(dates):
    # Weekdays from September to June (we ignore holidays for now)
    return (dates.dt.day_of_week < 5) & ~dates.dt.month.isin([7, 8])

df = add_hazards(df)
school = df[is_school_day(df["time"])].copy()
school["year"] = school["time"].dt.year

print("School days:", len(school))
print("Unsafe school days:", school["unsafe"].sum())
print()
print("Unsafe school days per year, by reason:")
print(school.groupby("year")[hazards + ["unsafe"]].sum().T)   # .T flips the table so it fits

## Step 3: Turn each day into a question about *tomorrow*

Right now each row is one day. We want: "using **today's** weather and air, predict **tomorrow's** dangers."

- `features` = what we know today (the model's inputs).
- `shift(-1)` moves tomorrow's answers up one row, so each row holds today's inputs **and** tomorrow's answers side by side.
- We keep only rows where **tomorrow is a school day**.

**Leakage check:** the inputs are only from today. Nothing from tomorrow goes into `features`.

**Wind direction trick:** direction is an angle, so 359° and 1° are almost the same direction even though the numbers are far apart. We turn the angle into two numbers, `sin` and `cos`, which are close together for directions that are close together. That way the model can learn things like "wind from the southwest brings smog".

**Pollution inputs:** besides PM2.5 and ozone, the model also sees today's **PM10, nitrogen dioxide and US AQI**.

In [ ]:
df = df.sort_values("time").reset_index(drop=True)

import numpy as np

def add_features(t):
    t["month"] = t["time"].dt.month
    angle = np.radians(t["wind_direction_10m_dominant"])
    t["wind_dir_sin"] = np.sin(angle)
    t["wind_dir_cos"] = np.cos(angle)
    return t

df = add_features(df)
features = ["apparent_temperature_max", "apparent_temperature_min", "precipitation_sum",
            "snowfall_sum", "wind_gusts_10m_max", "wind_speed_10m_max", "wind_dir_sin", "wind_dir_cos",
            "weather_code", "pm2_5", "ozone_8h", "pm10", "nitrogen_dioxide", "us_aqi", "month"]

# Tomorrow's answers, moved up one row
target_cols = []
for h in hazards:
    df[h + "_tomorrow"] = df[h].shift(-1)
    target_cols.append(h + "_tomorrow")

df["tomorrow"] = df["time"] + pd.Timedelta(days=1)
tomorrow_is_school = is_school_day(df["tomorrow"])

data = df[tomorrow_is_school].dropna(subset=features + target_cols).copy()
data[target_cols] = data[target_cols].astype(bool)
data["unsafe_tomorrow"] = data[target_cols].any(axis=1)
print("Rows we can learn from:", len(data))

## Step 4: Split by time (never shuffle)

- **Train** on every school day **before** September 2025.
- **Test** on the **2025-26 school year**, which the model never sees while learning.

This is like a real forecast: you only ever know the past.

In [ ]:
TEST_START = "2025-09-01"
train = data[data["tomorrow"] < TEST_START]
test = data[data["tomorrow"] >= TEST_START]

print("Train:", train["tomorrow"].min().date(), "to", train["tomorrow"].max().date(), "-", len(train), "days")
print("Test: ", test["tomorrow"].min().date(), "to", test["tomorrow"].max().date(), "-", len(test), "days")
print("Unsafe share in train:", round(train["unsafe_tomorrow"].mean(), 2))
print("Unsafe share in test: ", round(test["unsafe_tomorrow"].mean(), 2))

## Step 4b: What brings pollution to Philadelphia?

Before predicting, we look at **what pollution goes along with** in the training years only (the test year stays untouched):

1. **Pollution rose:** the average AQI and PM2.5 for each **wind direction**. If wind from one direction brings dirtier air, that direction's bar is taller.
2. **Correlation:** a number from -1 to 1 for how AQI moves with each weather input. Near 1 means "goes up together," near -1 means "one goes up, the other goes down," and near 0 means "no clear link."

**Correlation is not cause.** Hot days and high ozone go together because sunlight makes ozone, but a number alone can't prove that. We use the results to explain, not to claim causes we didn't test.

AQI categories come from the US EPA: 0-50 Good, 51-100 Moderate, 101-150 Unhealthy for sensitive groups (which includes children), 151-200 Unhealthy, 201-300 Very unhealthy, 301+ Hazardous.

In [ ]:
import matplotlib.pyplot as plt

COMPASS = ["N", "NE", "E", "SE", "S", "SW", "W", "NW"]
def compass(deg):
    return COMPASS[int(((deg % 360) + 22.5) // 45) % 8]

past = df[df["time"] < TEST_START].dropna(subset=["wind_direction_10m_dominant", "us_aqi"]).copy()
past["wind_from"] = past["wind_direction_10m_dominant"].map(compass)
rose = past.groupby("wind_from")[["us_aqi", "pm2_5"]].mean().reindex(COMPASS).round(1)
rose["days"] = past["wind_from"].value_counts().reindex(COMPASS)
print("Average pollution by wind direction (training years):")
print(rose)

links = past[["us_aqi", "apparent_temperature_max", "wind_speed_10m_max", "precipitation_sum", "month"]].corr()["us_aqi"]
print()
print("How AQI moves with each weather input (correlation, -1 to 1):")
print(links.drop("us_aqi").round(2).to_string())
if "main_pollutant" in past:
    print()
    print("Main pollutant, share of days:")
    print(past["main_pollutant"].value_counts(normalize=True).round(2).to_string())

plt.figure(figsize=(7, 3.5))
plt.bar(rose.index, rose["us_aqi"], color="#0F2A43")
plt.ylabel("Average daily US AQI")
plt.xlabel("Wind comes from")
plt.title("Philadelphia: average air quality by wind direction (2022-25)")
plt.savefig("pollution_by_wind.png", dpi=150, bbox_inches="tight")
plt.show()

## Step 5: The baseline, "tomorrow will be the same as today"

The simplest possible guess: whatever danger happened today, guess it happens again tomorrow. Our model is only useful if it beats this.

In [ ]:
baseline_pred = test[hazards].values.astype(bool)   # today's dangers = guess for tomorrow

## Step 6: Train the model (a random forest)

A **random forest** is a group of decision trees. Each tree asks yes/no questions like "is today's 'feels like' max above 30 °C?" and the trees vote.

- We train **one forest that predicts all 9 dangers at once** (scikit-learn allows several answers per row).
- `class_weight="balanced"` tells the model that rare unsafe days matter, so it doesn't just say "safe" every time.
- `random_state=42` makes the result the same every time we run it.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(n_estimators=200, min_samples_leaf=3,
                               class_weight="balanced", random_state=42)
model.fit(train[features], train[target_cols])
model_pred = model.predict(test[features]).astype(bool)

## Step 7: Model vs. baseline

For each danger we count, on the test year:
- **caught**: dangerous days we warned about (higher is better)
- **missed**: dangerous days we did NOT warn about (the worst mistake for kids)
- **false alarms**: we warned but it was fine (annoying, less harmful)

⚠️ If accuracy is above 95%, be suspicious and look for leakage.

In [ ]:
y_true = test[target_cols].values.astype(bool)

def score(true, pred):
    return {"caught": int((true & pred).sum()),
            "missed": int((true & ~pred).sum()),
            "false alarms": int((~true & pred).sum())}

rows = []
for i, h in enumerate(hazards):
    b = score(y_true[:, i], baseline_pred[:, i])
    m = score(y_true[:, i], model_pred[:, i])
    rows.append({"danger": h, "real days": int(y_true[:, i].sum()),
                 "baseline caught": b["caught"], "baseline missed": b["missed"], "baseline false alarms": b["false alarms"],
                 "model caught": m["caught"], "model missed": m["missed"], "model false alarms": m["false alarms"]})
print(pd.DataFrame(rows).to_string(index=False))

# Overall: is tomorrow unsafe for ANY reason?
true_any = y_true.any(axis=1)
base_any = baseline_pred.any(axis=1)
model_any = model_pred.any(axis=1)
print()
for name, pred in [("Baseline", base_any), ("Model", model_any)]:
    s = score(true_any, pred)
    acc = (pred == true_any).mean()
    print(f"{name}: accuracy {acc:.2f} | caught {s['caught']} | missed {s['missed']} | false alarms {s['false alarms']}")
    if acc > 0.95:
        print("   WARNING: above 95%. Check for leakage before trusting this!")

## Step 7b: Too many false alarms, so make the model less jumpy

**What we saw on our first run:** the model missed only 4 unsafe days, but raised 100 false alarms. No principal would trust a tool that cancels recess most days.

**The fix:** for each danger, the model gives a **chance from 0 to 1**. Until now it warned whenever the chance was above 0.5. We'll pick a better **cutoff**:

1. Hold back the **2024-25 school year** from the training data as a practice year ("validation").
2. Train a model on the years before it, and try cutoffs from 0.05 to 0.95.
3. Keep the **lowest cutoff whose false alarms are no more than the baseline's** on that year. Lowest means it catches as many unsafe days as possible.
4. Use that cutoff on the real test year (2025-26) **once**.

**Why not pick the cutoff using the test year?** That would be cheating: we'd be tuning on the answers. The test year stays untouched until the end.

In [ ]:
import numpy as np

def danger_probs(m, X):
    # The model's chance (0 to 1) of each danger, one column per danger
    cols = []
    for p, classes in zip(m.predict_proba(X), m.classes_):
        classes = list(classes)
        cols.append(p[:, classes.index(True)] if True in classes else np.zeros(len(X)))
    return np.column_stack(cols)

VAL_START = "2024-09-01"
fit_part = train[train["tomorrow"] < VAL_START]
val_part = train[train["tomorrow"] >= VAL_START]

tuner = RandomForestClassifier(n_estimators=200, min_samples_leaf=3,
                               class_weight="balanced", random_state=42)
tuner.fit(fit_part[features], fit_part[target_cols])

val_true = val_part[target_cols].values.astype(bool).any(axis=1)
val_base = val_part[hazards].values.astype(bool).any(axis=1)
val_prob = danger_probs(tuner, val_part[features]).max(axis=1)   # chance of the likeliest danger
base_false_alarms = int((~val_true & val_base).sum())

THRESHOLD = 0.95                                  # fallback: very cautious
for t in np.arange(0.05, 1.0, 0.05):
    if (~val_true & (val_prob >= t)).sum() <= base_false_alarms:
        THRESHOLD = round(float(t), 2)
        break
print("Baseline false alarms on the practice year:", base_false_alarms)
print("Chosen cutoff:", THRESHOLD)

# Use the cutoff on the real test year, with the model trained on all training years
model_pred = danger_probs(model, test[features]) >= THRESHOLD
model_any = model_pred.any(axis=1)
for name, pred in [("Baseline", base_any), ("Model (tuned)", model_any)]:
    s = score(true_any, pred)
    acc = (pred == true_any).mean()
    print(f"{name}: accuracy {acc:.2f} | caught {s['caught']} | missed {s['missed']} | false alarms {s['false alarms']}")
    if acc > 0.95:
        print("   WARNING: above 95%. Check for leakage before trusting this!")

From here on, every step uses the **tuned** model.

## Step 8: Confusion matrix (overall "unsafe tomorrow")

A 2x2 table: rows are what really happened, columns are what the model said.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

ConfusionMatrixDisplay.from_predictions(true_any, model_any, display_labels=["Safe", "Unsafe"], cmap="Blues")
plt.title("Recess Ready: model on the 2025-26 school year")
plt.savefig("confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

## Step 9: Where does it fail?

Real unsafe days the model **missed**. We look at a few and ask why.

In [ ]:
missed = test[true_any & ~model_any]
print("Missed unsafe days:", len(missed))
for _, row in missed.head(3).iterrows():
    real = [h for h in hazards if row[h + "_tomorrow"]]
    print(f"- {row['tomorrow'].date()}: really {real}. "
          f"Today's feels-like max {row['apparent_temperature_max']:.0f} C, min {row['apparent_temperature_min']:.0f} C, "
          f"PM2.5 {row['pm2_5']:.0f}, gusts {row['wind_gusts_10m_max']:.0f} km/h")

## Step 10: Does it work equally well in every season?

Our fairness check: the same accuracy and missed-day counts, split by season.

In [ ]:
def season(month):
    if month in [9, 10, 11]: return "Fall"
    if month in [12, 1, 2]: return "Winter"
    if month in [3, 4, 5]: return "Spring"
    return "June"

seasons = test["tomorrow"].dt.month.map(season).values
rows = []
for s in ["Fall", "Winter", "Spring", "June"]:
    keep = seasons == s
    t, m, b = true_any[keep], model_any[keep], base_any[keep]
    rows.append({"season": s, "days": int(keep.sum()), "unsafe days": int(t.sum()),
                 "model accuracy": round((m == t).mean(), 2),
                 "baseline accuracy": round((b == t).mean(), 2),
                 "model missed": int((t & ~m).sum())})
print(pd.DataFrame(rows).to_string(index=False))

## Step 10b: Does a Philadelphia model work across the whole region?

We trained only on Center City. Does it still work 20-60 km away, in New Jersey, Delaware and the Pennsylvania suburbs? This is our "does it hold up in a new place?" test, and it powers the regional danger chart in the app.

1. Make a **grid of 35 places** (every 0.2° of latitude and longitude) around Philadelphia.
2. Download the test year for all 35 places **in one request**, then reuse `daily_table`, `add_hazards` and `add_features`, so every place follows the exact same rules.
3. Run the **same model and cutoff** (no retraining) and score each place.

Weak spots on this list are real failures to report, not to hide.

In [ ]:
REG_LATS = [39.6, 39.8, 40.0, 40.2, 40.4]
REG_LONS = [-75.8, -75.6, -75.4, -75.2, -75.0, -74.8, -74.6]
points = [(la, lo) for la in REG_LATS for lo in REG_LONS]

REG_START = (pd.Timestamp(TEST_START) - pd.Timedelta(days=10)).strftime("%Y-%m-%d")
def region_table():
    reg_weather, reg_air = download([p[0] for p in points], [p[1] for p in points], REG_START, END)
    parts = []
    for (la, lo), w, a in zip(points, reg_weather, reg_air):
        part = daily_table(w, a)
        part["lat"], part["lon"] = la, lo
        parts.append(part)
    return pd.concat(parts)

region_raw = cached("region_daily", region_table)

reg_tables = []
for (la, lo), part in region_raw.groupby(["lat", "lon"]):
    t = add_features(add_hazards(part.sort_values("time").reset_index(drop=True)))
    t["tomorrow"] = t["time"] + pd.Timedelta(days=1)
    for h in hazards:
        t[h + "_tomorrow"] = t[h].shift(-1)
    reg_tables.append(t)
region = pd.concat(reg_tables)
region = region[is_school_day(region["tomorrow"]) & (region["tomorrow"] >= TEST_START)]
region = region.dropna(subset=features + target_cols).copy()
region[target_cols] = region[target_cols].astype(bool)

# One prediction for every place and day at once (much faster than one at a time)
reg_probs = danger_probs(model, region[features])
region["warn"] = (reg_probs >= THRESHOLD).any(axis=1)
region["really_unsafe"] = region[target_cols].any(axis=1)

rows = []
for (la, lo), g in region.groupby(["lat", "lon"]):
    rows.append({"lat": la, "lon": lo, "accuracy": round((g["warn"] == g["really_unsafe"]).mean(), 2),
                 "unsafe days": int(g["really_unsafe"].sum()),
                 "missed": int((g["really_unsafe"] & ~g["warn"]).sum())})
place_scores = pd.DataFrame(rows)
print("Accuracy across the 35 places: from", place_scores["accuracy"].min(), "to", place_scores["accuracy"].max())
print("Philadelphia (Center City) accuracy, for comparison:", round((model_any == true_any).mean(), 2))
print()
print("5 weakest places:")
print(place_scores.sort_values("accuracy").head(5).to_string(index=False))

## Step 10c: Philadelphia vs. Delhi

Delhi, India has some of the worst air in the world, and its schools have been closed because of smog. We run the **exact same pipeline** there:

1. Download Delhi's weather and air for the same dates (in India's time zone).
2. Use the same danger rules. School days: Monday-Friday, skipping **June**, Delhi's main summer break. This is a simplification; real school calendars vary, and the break often starts in mid-May.
3. Compare three things on Delhi's test year:
   - "Tomorrow = today" for Delhi
   - **A Delhi model** trained on Delhi's earlier years (same settings as Philadelphia's)
   - **The Philadelphia model** used in Delhi without retraining. Does it hold up somewhere very different?

The Delhi model gets its own cutoff, chosen on Delhi's 2024-25 practice year, the same way as Step 7b.

In [ ]:
DELHI_LAT, DELHI_LON = 28.61, 77.21
def delhi_table():
    dw, da = download([DELHI_LAT], [DELHI_LON], START, END, tz="Asia/Kolkata")
    return daily_table(dw[0], da[0])

delhi = add_features(add_hazards(cached("delhi_daily", delhi_table))).sort_values("time").reset_index(drop=True)
delhi_all = delhi.copy()          # every day, for the month-by-month AQI comparison
for h in hazards:
    delhi[h + "_tomorrow"] = delhi[h].shift(-1)
delhi["tomorrow"] = delhi["time"] + pd.Timedelta(days=1)
delhi_school = (delhi["tomorrow"].dt.day_of_week < 5) & (delhi["tomorrow"].dt.month != 6)
delhi = delhi[delhi_school].dropna(subset=features + target_cols).copy()
delhi[target_cols] = delhi[target_cols].astype(bool)

d_train = delhi[delhi["tomorrow"] < TEST_START]
d_test = delhi[delhi["tomorrow"] >= TEST_START]
d_true = d_test[target_cols].values.any(axis=1)

# 1) Baseline
d_base = d_test[hazards].values.astype(bool).any(axis=1)

# 2) A Delhi model, with its cutoff picked on Delhi's practice year (same recipe as Step 7b)
d_fit, d_val = d_train[d_train["tomorrow"] < VAL_START], d_train[d_train["tomorrow"] >= VAL_START]
d_tuner = RandomForestClassifier(n_estimators=200, min_samples_leaf=3, class_weight="balanced", random_state=42)
d_tuner.fit(d_fit[features], d_fit[target_cols])
d_val_true = d_val[target_cols].values.any(axis=1)
d_val_prob = danger_probs(d_tuner, d_val[features]).max(axis=1)
d_base_fa = int((~d_val_true & d_val[hazards].values.astype(bool).any(axis=1)).sum())
D_THRESHOLD = 0.95
for t in np.arange(0.05, 1.0, 0.05):
    if (~d_val_true & (d_val_prob >= t)).sum() <= d_base_fa:
        D_THRESHOLD = round(float(t), 2)
        break
d_model = RandomForestClassifier(n_estimators=200, min_samples_leaf=3, class_weight="balanced", random_state=42)
d_model.fit(d_train[features], d_train[target_cols])
d_own = (danger_probs(d_model, d_test[features]) >= D_THRESHOLD).any(axis=1)

# 3) The Philadelphia model, used in Delhi with no retraining
d_philly = (danger_probs(model, d_test[features]) >= THRESHOLD).any(axis=1)

delhi_results = {"“Tomorrow = today”": d_base, "Delhi model": d_own, "Philadelphia model in Delhi": d_philly}
print("Delhi test year:", len(d_test), "school days,", int(d_true.sum()), "really unsafe")
for name, pred in delhi_results.items():
    s_ = score(d_true, pred)
    print(f"{name}: accuracy {(pred == d_true).mean():.2f} | caught {s_['caught']} | "
          f"missed {s_['missed']} | false alarms {s_['false alarms']}")

print()
print("Average daily US AQI by month (all years):")
aqi_by_month = pd.DataFrame({
    "Philadelphia": df.groupby(df["time"].dt.month)["us_aqi"].mean(),
    "Delhi": delhi_all.groupby(delhi_all["time"].dt.month)["us_aqi"].mean()}).round(0)
print(aqi_by_month.T.to_string())

## Step 11: The demo app

Pick a school day from the test year. The app shows what Recess Ready would have told the school **the evening before**, and what really happened.

The app has four pieces, each made by one small function that returns a bit of web page (HTML) as text:
- `meters_html()`: a bar for each danger showing the model's **chance** (0-100%), with our cutoff marked. A bar past the line means a warning.
- `map_html()`: a Life360-style map of Philadelphia. `build_base_map()` downloads small map pictures ("tiles") from CARTO **once** and glues them into one picture. `to_pixel()` uses standard map math (Web Mercator) to turn latitude/longitude into a spot on that picture, so we can place the school pin, the wind pin and the dashed path. Open-Meteo gives one reading for the whole city, so we show one point, not a fake reading for each neighborhood.
- `week_html()`: two small charts of the **7 days before the call**, feels-like temperature and PM2.5, against their danger lines.
- `chart_html()`: a **regional danger chart** in the style of the National Weather Service forecast chart. Each square is one of the 35 places from Step 10b, shaded by its most likely danger if that chance passes our cutoff (stripes = NWS-style "possible" areas), with a wind arrow at each place. Unlike the NWS chart, we don't draw fronts or high/low pressure: our data can't support that honestly.
- `air_html()`: today's **US AQI** in the official EPA colors, what the level means for kids (EPA wording, simplified), the **main pollutant**, and today's PM2.5, PM10, ozone and NO₂.
- `rose_html()`: the **pollution rose** from Step 4b, with tomorrow's wind direction highlighted.
- `delhi_html()`: the **Philadelphia vs. Delhi** results from Step 10c.
- `recess_check()` puts it together for the chosen date.

*Prototype, not official guidance. Always follow official National Weather Service alerts and your school district.*

In [ ]:
import gradio as gr
import html

NAMES = {"too_hot": ("🔥", "Too hot"), "too_cold": ("🥶", "Too cold"),
         "bad_air": ("😷", "Smoke / particles"), "ozone_smog": ("🌫️", "Ozone smog"),
         "heavy_rain": ("🌊", "Flash-flood rain"), "high_wind": ("🌀", "Dangerous wind"),
         "thunderstorm": ("⚡", "Lightning"), "heavy_snow": ("❄️", "Heavy snow"),
         "ice": ("🧊", "Ice")}
COMPASS = ["N", "NE", "E", "SE", "S", "SW", "W", "NW"]

# Real test-year scores (calculated from the results above, never typed in by hand)
m_score = score(true_any, model_any)
b_score = score(true_any, base_any)
total_unsafe = int(true_any.sum())
test_dates = test["tomorrow"].dt.strftime("%Y-%m-%d").tolist()


# ---------- Piece 1: danger meters (the model's chance for each danger) ----------
def meters_html(chances, happened):
    rows = ""
    order = sorted(range(len(hazards)), key=lambda i: -chances[i])   # most likely first
    for i in order:
        h = hazards[i]
        icon, name = NAMES[h]
        pct = round(chances[i] * 100)
        hot = "hot" if chances[i] >= THRESHOLD else ""
        tag = '<span class="tag">happened</span>' if h in happened else ""
        rows += (f'<div class="meter {hot}"><span class="mname">{icon} {name} {tag}</span>'
                 f'<span class="track"><span class="fill" style="width:{pct}%"></span>'
                 f'<span class="cut" style="left:{THRESHOLD*100:.0f}%"></span></span>'
                 f'<span class="pct">{pct}%</span></div>')
    return (f'<div class="board"><p class="board-head">Chance of each danger tomorrow. '
            f'The line marks our cutoff ({THRESHOLD*100:.0f}%).</p>{rows}</div>')


# ---------- Piece 2: a Life360-style map with the wind ----------
# We download small square map pictures ("tiles") once, glue them into one picture,
# then draw pins on top with HTML. No map library is needed, so nothing can block it.
import base64, io, math
from PIL import Image

TILE = 512                           # tile size in pixels (512 = sharp "@2x" tiles)
ZOOM = 11                            # zoom level for the close-up map
MAP_W, MAP_H = 1200, 1000             # size of the glued-together map picture

def to_pixel(lat, lon, zoom=ZOOM):
    # Standard "Web Mercator" math: turns latitude/longitude into a pixel on the world map
    n = TILE * 2 ** zoom
    x = (lon + 180) / 360 * n
    y = (1 - math.log(math.tan(math.radians(lat)) + 1 / math.cos(math.radians(lat))) / math.pi) / 2 * n
    return x, y

cx, cy = to_pixel(LAT, LON)
left, top = cx - MAP_W / 2, cy - MAP_H * 0.42   # put the school a bit above the middle

def stitch(zoom, left, top, width, height):
    # Download the map tiles that cover this box and glue them into one picture
    img = Image.new("RGB", (width, height), (221, 231, 239))
    try:
        for tx in range(int(left // TILE), int((left + width) // TILE) + 1):
            for ty in range(int(top // TILE), int((top + height) // TILE) + 1):
                url = f"https://a.basemaps.cartocdn.com/rastertiles/voyager/{zoom}/{tx}/{ty}@2x.png"
                r = requests.get(url, timeout=10, headers={"User-Agent": "RecessReady student project"})
                r.raise_for_status()
                tile = Image.open(io.BytesIO(r.content)).convert("RGB")
                img.paste(tile, (int(tx * TILE - left), int(ty * TILE - top)))
        ok = True
    except Exception as err:
        print("Map tiles could not be downloaded, showing a plain map instead:", err)
        ok = False
    buf = io.BytesIO()
    img.save(buf, format="JPEG", quality=82)
    return base64.b64encode(buf.getvalue()).decode(), ok

BASE_MAP, MAP_OK = stitch(ZOOM, left, top, MAP_W, MAP_H)   # downloaded once, reused for every date

def map_html(wind_from, gust, state, pm, feels_max):
    label = COMPASS[int(((wind_from % 360) + 22.5) // 45) % 8]
    # Upwind point: about 10 km in the direction the wind comes FROM
    ux, uy = to_pixel(LAT + 0.09 * math.cos(math.radians(wind_from)),
                      LON + 0.09 * math.sin(math.radians(wind_from)) / math.cos(math.radians(LAT)))
    hx, hy = cx - left, cy - top          # school point, in map-picture pixels
    ux, uy = ux - left, uy - top
    pct = lambda x, y: f"left:{x / MAP_W * 100:.2f}%;top:{y / MAP_H * 100:.2f}%"
    pill = "Indoor recess" if state == "indoor" else "Outdoor recess"
    note = "" if MAP_OK else '<span class="mapnote">Map pictures could not load. Pins are still in the right places.</span>'
    return f"""
    <div class="card mapcard"><p class="card-head">Where tomorrow's air comes from</p>
      <div class="mapwrap">
      <div class="lmap" style="aspect-ratio:{MAP_W}/{MAP_H};background-image:url(data:image/jpeg;base64,{BASE_MAP})">
        <svg class="lmap-svg" viewBox="0 0 {MAP_W} {MAP_H}" preserveAspectRatio="none">
          <line x1="{ux:.0f}" y1="{uy:.0f}" x2="{hx:.0f}" y2="{hy:.0f}" class="path"/>
        </svg>
        <div class="chip">🌬️ Wind from the {label}, gusts {gust:.0f} km/h</div>
        <div class="pin up" style="{pct(ux, uy)}"><span>🌬️</span></div>
        <div class="pin home" style="{pct(hx, hy)}"><span class="pulse"></span><span class="avatar">🏫</span></div>
        {note}
      </div>
        <div class="sheet"><span class="handle"></span>
          <div class="sheet-row"><div><b>Center City, Philadelphia</b>
            <p>Today: PM2.5 {pm:.0f} µg/m³, feels-like high {feels_max:.0f}°C</p></div>
            <span class="pill {state}">{pill}</span></div></div>
      </div>
      <p class="attr">Map © OpenStreetMap contributors © CARTO</p></div>"""


# ---------- Piece 3: small line charts of the week before ----------
def line_chart(values, lines, unit):
    # values: dict name -> (list of numbers, color); lines: list of (danger level, label)
    w, h, pad = 300, 120, 22
    allv = [v for vals, _ in values.values() for v in vals] + [lvl for lvl, _ in lines]
    lo, hi = min(allv), max(allv)
    span = (hi - lo) or 1
    x = lambda i: pad + i * (w - 2 * pad) / 6
    y = lambda v: h - pad - (v - lo) / span * (h - 2 * pad)
    svg = f'<svg viewBox="0 0 {w} {h}" class="chart" role="img">'
    for lvl, lab in lines:
        svg += (f'<line x1="{pad}" x2="{w-pad}" y1="{y(lvl):.1f}" y2="{y(lvl):.1f}" class="danger"/>'
                f'<text x="{w-pad}" y="{y(lvl)-4:.1f}" text-anchor="end" class="dlab">{lab}</text>')
    for name, (vals, color) in values.items():
        pts = " ".join(f"{x(i):.1f},{y(v):.1f}" for i, v in enumerate(vals))
        svg += f'<polyline points="{pts}" fill="none" stroke="{color}" stroke-width="2.5"/>'
        svg += f'<circle cx="{x(len(vals)-1):.1f}" cy="{y(vals[-1]):.1f}" r="3.5" fill="{color}"/>'
        svg += (f'<text x="{x(len(vals)-1)+6:.1f}" y="{y(vals[-1])+4:.1f}" class="vlab" fill="{color}">'
                f'{vals[-1]:.0f}{unit}</text>')
    svg += f'<text x="{pad}" y="{h-4}" class="axis">7 days ago</text><text x="{w-pad}" y="{h-4}" text-anchor="end" class="axis">today</text>'
    return svg + "</svg>"

def week_html(tomorrow):
    week = df[(df["time"] > tomorrow - pd.Timedelta(days=8)) & (df["time"] < tomorrow)].tail(7)
    temp = line_chart({"max": (week["apparent_temperature_max"].tolist(), "#C77C00"),
                       "min": (week["apparent_temperature_min"].tolist(), "#0F2A43")},
                      [(32, "too hot"), (-7, "too cold")], "°")
    air = line_chart({"pm": (week["pm2_5"].tolist(), "#C77C00")}, [(35.4, "unhealthy")], "")
    return (f'<div class="card"><p class="card-head">What Recess Ready saw this week</p>'
            f'<p class="chart-title">Feels-like temperature, °C (amber = daily high, navy = daily low)</p>{temp}'
            f'<p class="chart-title">PM2.5 particles, µg/m³</p>{air}</div>')


# ---------- Piece 5: today's air quality (official US EPA AQI scale and colors) ----------
AQI_LEVELS = [(50, "Good", "#00E400", "Air is fine for outdoor play."),
              (100, "Moderate", "#FFFF00", "Air is fine for most kids. A few very sensitive kids may notice it."),
              (150, "Unhealthy for sensitive groups", "#FF7E00",
               "Kids with asthma or other lung problems should limit long or hard outdoor play."),
              (200, "Unhealthy", "#FF0000", "Everyone should limit long or hard outdoor play."),
              (300, "Very unhealthy", "#8F3F97", "Everyone should avoid long or hard outdoor play."),
              (9999, "Hazardous", "#7E0023", "Everyone should stay indoors.")]
POLLUTANT_NAMES = {"pm2_5": "PM2.5 particles", "pm10": "PM10 particles", "ozone": "ozone",
                   "nitrogen_dioxide": "nitrogen dioxide", "sulphur_dioxide": "sulfur dioxide",
                   "carbon_monoxide": "carbon monoxide"}

def aqi_level(value):
    for top, name, color, advice in AQI_LEVELS:
        if value <= top:
            return name, color, advice

def air_html(row):
    aqi = row["us_aqi"]
    name, color, advice = aqi_level(aqi)
    main = POLLUTANT_NAMES.get(row.get("main_pollutant", ""), "not available")
    text_color = "#0F2A43" if name in ("Good", "Moderate", "Unhealthy for sensitive groups") else "#FFFFFF"
    # Draw the scale to size: each color band is as wide as its AQI range (the 200-300 band is twice as wide)
    bars, low = "", 0
    for top, lvl, col, _ in AQI_LEVELS[:5]:
        bars += f'<span style="background:{col};flex:{top - low}" title="{lvl}"></span>'
        low = top
    marker = min(aqi, 300) / 300 * 100
    ticks = "".join(f'<span style="left:{v / 300 * 100:.1f}%">{v}</span>' for v in [0, 50, 100, 150, 200, 300])
    return f"""
    <div class="card"><p class="card-head">Air quality today (US AQI)</p>
      <div class="aqi-top"><span class="aqi-num" style="background:{color};color:{text_color}">{aqi:.0f}</span>
        <div><b class="aqi-name">{name}</b><p class="aqi-advice">{advice}</p></div></div>
      <div class="aqi-scale">{bars}<i style="left:{marker:.0f}%"></i></div>
      <div class="aqi-scale-l">{ticks}</div>
      <p class="aqi-main">Main pollutant today: <b>{main}</b></p>
      <ul class="pollutants">
        <li><span>PM2.5</span><b>{row["pm2_5"]:.0f}</b> µg/m³</li><li><span>PM10</span><b>{row["pm10"]:.0f}</b> µg/m³</li>
        <li><span>Ozone</span><b>{row["ozone_8h"]:.0f}</b> µg/m³</li><li><span>NO₂</span><b>{row["nitrogen_dioxide"]:.0f}</b> µg/m³</li>
      </ul></div>"""

# ---------- Piece 6: pollution rose (average AQI by wind direction, training years) ----------
def rose_html(wind_from):
    today = compass(wind_from)
    biggest = rose["us_aqi"].max()
    rows_h = ""
    for d in COMPASS:
        v = rose.loc[d, "us_aqi"]
        cls = "rose-row now" if d == today else "rose-row"
        rows_h += (f'<div class="{cls}"><span class="rd">{d}</span><span class="rtrack"><span style="width:{v / biggest * 100:.0f}%"></span></span>'
                   f'<span class="rv">{v:.0f}</span></div>')
    return f"""
    <div class="card"><p class="card-head">What brings pollution to Philadelphia?</p>
      <p class="chart-title">Average AQI by the direction the wind comes from (2022-25). Tomorrow's wind: <b>from the {today}</b>.</p>
      {rows_h}
      <p class="chart-title">Correlation shows what goes together, not what causes it.</p></div>"""


# ---------- Piece 4: regional danger chart (in the style of the NWS forecast chart) ----------
CZ = 9                                                       # zoomed out to show the whole region
c_left, c_top = to_pixel(max(REG_LATS) + 0.22, min(REG_LONS) - 0.3, CZ)
c_right, c_bottom = to_pixel(min(REG_LATS) - 0.22, max(REG_LONS) + 0.3, CZ)
CH_W, CH_H = int(c_right - c_left), int(c_bottom - c_top)
CHART_MAP, CHART_OK = stitch(CZ, c_left, c_top, CH_W, CH_H)

# NWS-like colors. True = striped (like "Possible" areas on the NWS chart)
STYLE = {"too_hot": ("#E4572E", True), "too_cold": ("#5DA9E9", False), "bad_air": ("#7B5EA7", False),
         "ozone_smog": ("#B08FD8", True), "heavy_rain": ("#D7263D", True), "high_wind": ("#B8860B", False),
         "thunderstorm": ("#E6C700", True), "heavy_snow": ("#8A99A8", True), "ice": ("#E26BB5", True)}

for i, h in enumerate(hazards):                 # keep each danger's chance next to its row
    region[h + "_chance"] = reg_probs[:, i]

def chart_html(tomorrow):
    rows = region[region["tomorrow"] == tomorrow]
    defs, cells_svg, arrows, seen = "", "", "", []
    for h, (color, striped) in STYLE.items():
        if striped:
            defs += (f'<pattern id="p_{h}" width="16" height="16" patternUnits="userSpaceOnUse" '
                     f'patternTransform="rotate(45)"><rect width="16" height="16" fill="{color}" fill-opacity=".35"/>'
                     f'<rect width="6" height="16" fill="{color}"/></pattern>')
    for _, r in rows.iterrows():
        chances = {h: r[h + "_chance"] for h in hazards}
        top_h = max(chances, key=chances.get)       # the most likely danger at this place
        x0, y0 = to_pixel(r["lat"] + 0.1, r["lon"] - 0.1, CZ)
        x1, y1 = to_pixel(r["lat"] - 0.1, r["lon"] + 0.1, CZ)
        x0, y0, x1, y1 = x0 - c_left, y0 - c_top, x1 - c_left, y1 - c_top
        if chances[top_h] >= THRESHOLD:
            color, striped = STYLE[top_h]
            fill = f"url(#p_{top_h})" if striped else color
            name = NAMES[top_h][1]
            cells_svg += (f'<rect x="{x0:.0f}" y="{y0:.0f}" width="{x1-x0:.0f}" height="{y1-y0:.0f}" fill="{fill}" '
                          f'stroke="none"><title>{name}: {chances[top_h]*100:.0f}% chance</title></rect>')
            if top_h not in seen:
                seen.append(top_h)
        # wind arrow at the middle of the square, pointing where the wind blows TO
        mx, my = (x0 + x1) / 2, (y0 + y1) / 2
        length = 14 + min(r["wind_gusts_10m_max"], 80) * 0.35
        arrows += (f'<g transform="translate({mx:.0f},{my:.0f}) rotate({(r["wind_direction_10m_dominant"] + 180) % 360:.0f})">'
                   f'<line x1="0" y1="{length/2:.0f}" x2="0" y2="{-length/2:.0f}" class="warrow"/>'
                   f'<path d="M0,{-length/2-7:.0f} L5,{-length/2+2:.0f} L-5,{-length/2+2:.0f} Z" class="whead"/></g>')
    px, py = to_pixel(LAT, LON, CZ)
    px, py = px - c_left, py - c_top
    legend = "".join(
        f'<li><span class="sw" style="background:{STYLE[h][0]}{"" if not STYLE[h][1] else ";background-image:repeating-linear-gradient(45deg,rgba(255,255,255,.55) 0 4px,transparent 4px 9px)"}"></span>{NAMES[h][1]}</li>'
        for h in hazards)
    day = pd.Timestamp(tomorrow).strftime("%A, %B %-d, %Y")
    summary = ("No dangers expected anywhere in the region." if not seen else
               "Dangers expected: " + ", ".join(NAMES[h][1].lower() for h in seen) + ".")
    note = "" if CHART_OK else '<span class="mapnote">Map pictures could not load. Shaded areas are still in the right places.</span>'
    return f"""
    <section class="chartsec"><h3>Regional danger chart</h3>
      <p class="chartsub">Our Philadelphia model, run on 35 places across the region for {day}. {summary}</p>
      <div class="nws">
        <div class="nwstitle">Recess Ready danger chart<br><span>Valid for school on {day}</span></div>
        <div class="nwsmap" style="aspect-ratio:{CH_W}/{CH_H};background-image:url(data:image/jpeg;base64,{CHART_MAP})">
          <svg viewBox="0 0 {CH_W} {CH_H}" preserveAspectRatio="none" class="nwssvg">
            <defs>{defs}</defs><g opacity=".8">{cells_svg}</g>{arrows}
            <circle cx="{px:.0f}" cy="{py:.0f}" r="9" fill="#0F2A43" stroke="#fff" stroke-width="3"/>
            <text x="{px + 14:.0f}" y="{py + 5:.0f}" class="cityl">Philadelphia</text>
          </svg>
          {note}
        </div>
        <ul class="legend">{legend}<li class="lg-note"><span class="arrow-sw">➝</span>Wind direction (longer = stronger gusts)</li></ul>
      </div>
      <p class="attr">Map © OpenStreetMap contributors © CARTO. Prototype, not official guidance; for official forecasts see weather.gov.</p>
    </section>"""


# ---------- Putting it together for one date ----------
def recess_check(date_str):
    idx = test.index[test["tomorrow"].dt.strftime("%Y-%m-%d") == date_str][0]
    row = test.loc[idx]
    chances = danger_probs(model, test.loc[[idx], features])[0]
    warned = [h for h, c in zip(hazards, chances) if c >= THRESHOLD]
    happened = [h for h in hazards if row[h + "_tomorrow"]]
    day = row["tomorrow"].strftime("%A, %B %-d")

    if warned:
        state, call = "indoor", "Plan indoor recess"
        why = "Watch out for " + ", ".join(NAMES[h][1].lower() for h in warned) + "."
    else:
        state, call = "outdoor", "Outdoor recess is on"
        why = "No dangers expected tomorrow."
    if warned and happened and not set(warned) & set(happened):
        verdict = "Right that it was unsafe, but for the wrong reason."
    elif (len(warned) > 0) == (len(happened) > 0):
        verdict = "Recess Ready got this day right."
    elif happened:
        verdict = "Missed. Recess Ready said it was safe."
    else:
        verdict = "False alarm. The day was actually safe."
    truth = ", ".join(NAMES[h][1].lower() for h in happened) if happened else "a safe day"

    return f"""
    <div class="result">
      <div class="call-side">
        <p class="kicker">The call for {day}</p>
        <h2 class="call {state}"><span>{call}</span></h2>
        <p class="why">{why}</p>
        <div class="truth"><p>What really happened: <b>{truth}</b></p><p class="verdict">{verdict}</p></div>
      </div>
      {meters_html(chances, happened)}
    </div>
    <div class="panels">
      {map_html(row["wind_direction_10m_dominant"], row["wind_gusts_10m_max"], state, row["pm2_5"], row["apparent_temperature_max"])}
      {week_html(row["tomorrow"])}
    </div>
    <div class="panels">
      {air_html(row)}
      {rose_html(row["wind_direction_10m_dominant"])}
    </div>
    {chart_html(row["tomorrow"])}"""

HERO = """
<nav class="nav"><span class="logo"><span class="mark">R</span>Recess Ready</span>
  <span class="navlinks"><a href="#how">How it works</a><a href="#scores">Results</a></span></nav>
<section class="hero">
  <h1>The recess call,<br>made the <em>night before.</em></h1>
  <p class="lede">Heat, cold, smoke, storms and ice. Recess Ready checks nine dangers for Philadelphia
  schools and tells principals tonight whether tomorrow's recess should move inside.</p>
</section>"""

def delhi_html():
    rows_h = ""
    for name, pred in delhi_results.items():
        s_ = score(d_true, pred)
        cls = "ours" if name == "Delhi model" else "base"
        rows_h += (f'<div class="srow {cls}"><span class="sname">{name}</span>'
                   f'<span><b>{s_["caught"]}</b> caught</span><span><b>{s_["missed"]}</b> missed</span>'
                   f'<span><b>{s_["false alarms"]}</b> false alarms</span></div>')
    months = ["J", "F", "M", "A", "M", "J", "J", "A", "S", "O", "N", "D"]
    top = max(aqi_by_month.max().max(), 1)
    bars = ""
    for m in range(1, 13):
        ph, de = aqi_by_month.loc[m, "Philadelphia"], aqi_by_month.loc[m, "Delhi"]
        bars += (f'<div class="mcol"><div class="mbars"><span class="ph" style="height:{ph / top * 100:.0f}%" title="Philadelphia {ph:.0f}"></span>'
                 f'<span class="de" style="height:{de / top * 100:.0f}%" title="Delhi {de:.0f}"></span></div><i>{months[m-1]}</i></div>')
    return (f'<section class="scores delhi"><h3>Philadelphia vs. Delhi</h3>'
            f'<p>The same pipeline in Delhi, India, where smog has closed schools. Delhi test year: {int(d_true.sum())} of '
            f'{len(d_true)} school days were really unsafe.</p>{rows_h}'
            f'<p class="chart-title">Average daily US AQI by month: <span class="key ph"></span> Philadelphia '
            f'<span class="key de"></span> Delhi</p><div class="months">{bars}</div></section>')

def scores_html():
    def row(name, s, cls):
        return (f'<div class="srow {cls}"><span class="sname">{name}</span>'
                f'<span><b>{s["caught"]}</b> caught</span><span><b>{s["missed"]}</b> missed</span>'
                f'<span><b>{s["false alarms"]}</b> false alarms</span></div>')
    return (f'<section class="scores" id="scores"><h3>How it did on the 2025-26 school year</h3>'
            f'<p>{total_unsafe} school days were really unsafe. The model never saw this year while learning.</p>'
            + row("Recess Ready", m_score, "ours")
            + row("“Tomorrow = today” guess", b_score, "base")
            + f'<p class="regline">Across 35 places around Philadelphia, accuracy ranged from <b>{place_scores["accuracy"].min():.2f}</b> to <b>{place_scores["accuracy"].max():.2f}</b> (Center City: <b>{(model_any == true_any).mean():.2f}</b>).</p>'
            + '</section>' + delhi_html()
            + '<section class="how" id="how"><h3>How it works</h3>'
            '<p>Every evening we take today’s weather, wind and air in Philadelphia from Open-Meteo, and a random forest '
            'predicts the chance of each danger tomorrow. If any chance passes a cutoff we chose on last school year, '
            'the call is indoor recess. The map shows our single data point for the city: Open-Meteo’s data is too '
            'coarse to give each neighborhood its own reading.</p></section>'
            '<p class="fine">Prototype, not official guidance. Always follow National Weather Service alerts and your '
            'school district. Air data is a computer-model estimate (CAMS via Open-Meteo), not a sensor reading. '
            'Map © OpenStreetMap contributors © CARTO (Voyager style).</p>')

CSS = """
:root, .dark { --cream:#EAF2F8; --ink:#0F2A43; --orange:#F5B700; --amber:#C77C00; --green:#2E8B57; --paper:#FFFFFF; --line:#D3E0EA; --muted:#4A6278;
  --body-background-fill: var(--cream); --background-fill-primary: var(--paper);
  --block-background-fill: var(--paper); --input-background-fill: #FFFFFF;
  --body-text-color: var(--ink); --block-label-text-color: var(--muted); --border-color-primary: var(--line);
  --input-border-color: var(--line); --block-border-color: var(--line); color-scheme: light; }
html, body, gradio-app, .gradio-container, .main, .wrap { background: var(--cream) !important; }
body, .gradio-container { background: var(--cream) !important; color: var(--ink) !important;
  font-family: 'Geist', system-ui, sans-serif !important; }
html { font-size: clamp(15px, 0.3vw + 13.5px, 18px); }
.gradio-container { max-width: min(1320px, 94vw) !important; margin-inline: auto !important; padding-inline: clamp(12px, 3vw, 32px) !important; }
footer { display: none !important; }
.nav { display:flex; justify-content:space-between; align-items:center; padding: 0.4rem 0 1.6rem; }
.logo { display:flex; gap:0.55rem; align-items:center; font-weight:700; font-size:1.05rem; color:var(--ink); }
.mark { background:var(--ink); color:var(--cream); width:28px; height:28px; border-radius:7px;
  display:grid; place-items:center; font-size:0.9rem; }
.navlinks a { color:var(--muted) !important; text-decoration:none; margin-left:1.6rem; font-size:0.95rem; }
.hero h1 { font-size: clamp(2.6rem, 6vw, 4.6rem); line-height:1.0; letter-spacing:-0.035em; font-weight:700;
  color:var(--ink) !important; margin:0 0 1rem; }
.hero h1 em { font-family:'Instrument Serif', Georgia, serif; font-weight:400; letter-spacing:-0.01em;
  text-decoration: underline; text-decoration-color: var(--orange); text-decoration-thickness: 3px; text-underline-offset: 0.12em; }
.hero .lede { font-size:1.15rem; line-height:1.55; color:var(--muted) !important; max-width:56ch; margin:0 0 1.4rem; }
#picker { background:transparent !important; border:none !important; gap:0.8rem; align-items:end; }
#picker .block { border-radius:14px !important; }
#picker label, #picker label span, #picker [data-testid='block-info'] { color:var(--muted) !important; }
#go { background:var(--orange) !important; color:var(--ink) !important; border:none !important; border-radius:999px !important;
  font-weight:600 !important; font-size:1.05rem !important; min-height:52px; }
#go:hover { background:#E0A600 !important; }
.result { display:grid; grid-template-columns: 1fr 1.15fr; gap:1.4rem; margin-top:0.6rem; }
.call-side, .card { background:var(--paper); border:1px solid var(--line); border-radius:20px; padding:1.6rem; }
.kicker { color:var(--muted) !important; margin:0; font-size:0.98rem; }
.call { font-size: clamp(1.9rem, 3.2vw, 2.7rem); letter-spacing:-0.03em; line-height:1.05; margin:0.3rem 0 0.6rem; font-weight:700; }
.call.indoor { color:var(--ink) !important; } .call.indoor span { background: linear-gradient(transparent 58%, var(--orange) 58%);
  -webkit-box-decoration-break: clone; box-decoration-break: clone; }
.call.outdoor { color:var(--green) !important; }
.why { font-size:1.12rem; margin:0 0 1.4rem; color:var(--ink) !important; }
.truth { border-top:1px solid var(--line); padding-top:1rem; }
.truth p { margin:0.2rem 0; color:var(--muted) !important; } .truth b { color:var(--ink); }
.truth .verdict { color:var(--ink) !important; font-weight:600; }
.board { background:var(--ink); border-radius:20px; padding:1.4rem 1.5rem; box-shadow: 10px 10px 0 var(--orange); }
.board-head { color:#A9BDD0 !important; margin:0 0 0.8rem; font-size:0.92rem; }
.meter { display:grid; grid-template-columns: minmax(8.5rem, 11.5rem) 1fr 3rem; gap:0.7rem; align-items:center; padding:0.32rem 0; }
.meter span { color:#B9CBDB !important; font-size:0.92rem; }
.meter.hot .mname, .meter.hot .pct { color:#fff !important; font-weight:600; }
.track { position:relative; height:10px; background:#1C3A57; border-radius:99px; }
.fill { position:absolute; left:0; top:0; bottom:0; background:#5C7A96; border-radius:99px; }
.meter.hot .fill { background:var(--orange); }
.cut { position:absolute; top:-4px; bottom:-4px; width:2px; background:#FFFFFF; }
.pct { text-align:right; font-variant-numeric: tabular-nums; }
.tag { font-size:0.7rem !important; border:1px solid currentColor; border-radius:99px; padding:0 0.45em; margin-left:0.3em; }
.panels { display:grid; grid-template-columns: 1.25fr 1fr; gap:1.4rem; margin-top:1.4rem; }
.card-head { font-weight:600; margin:0 0 0.8rem; color:var(--ink) !important; font-size:1.05rem; }
.mapcard { padding:1.2rem; }
.mapwrap { position:relative; }
.lmap { position:relative; width:100%; border-radius:18px; overflow:hidden; background-size:100% 100%;
  box-shadow: inset 0 0 0 1px var(--line); }
.lmap-svg { position:absolute; inset:0; width:100%; height:100%; }
.lmap .path { stroke:#C77C00; stroke-width:5; stroke-dasharray:14 12; stroke-linecap:round; }
.chip { position:absolute; top:12px; left:12px; background:#fff; color:var(--ink) !important; border-radius:999px;
  padding:0.4rem 0.8rem; font-size:0.85rem; font-weight:600; box-shadow:0 4px 14px rgba(15,42,67,.18); }
.pin { position:absolute; transform:translate(-50%,-50%); }
.pin.up span { display:grid; place-items:center; width:34px; height:34px; border-radius:50%; background:#F5B700;
  border:3px solid #fff; box-shadow:0 4px 12px rgba(15,42,67,.3); font-size:1rem; }
.pin.home { transform:translate(-50%,-100%); }
.avatar { position:relative; display:grid; place-items:center; width:52px; height:52px; border-radius:50%;
  background:var(--ink); border:4px solid #fff; box-shadow:0 6px 18px rgba(15,42,67,.35); font-size:1.5rem; z-index:2; }
.avatar::after { content:""; position:absolute; bottom:-11px; left:50%; transform:translateX(-50%);
  border:8px solid transparent; border-top:9px solid #fff; }
.pulse { position:absolute; left:50%; bottom:-14px; width:26px; height:26px; margin-left:-13px; border-radius:50%;
  background:rgba(15,42,67,.25); animation: pulse 2s ease-out infinite; }
@keyframes pulse { from { transform:scale(.6); opacity:.9; } to { transform:scale(3.2); opacity:0; } }
@media (prefers-reduced-motion: reduce) { .pulse { animation:none; opacity:.4; } }
.sheet { position:absolute; left:10px; right:10px; bottom:10px; background:#fff; border-radius:18px;
  padding:0.5rem 1rem 0.8rem; box-shadow:0 -2px 18px rgba(15,42,67,.18); }
.handle { display:block; width:38px; height:4px; border-radius:99px; background:#C9D6E2; margin:0 auto 0.5rem; }
.sheet-row { display:flex; justify-content:space-between; align-items:center; gap:0.8rem; }
.sheet b { color:var(--ink) !important; font-size:0.98rem; } .sheet p { margin:0.1rem 0 0; color:var(--muted) !important; font-size:0.85rem; }
.pill { flex:none; border-radius:999px; padding:0.35rem 0.75rem; font-size:0.82rem; font-weight:700; }
.pill.indoor { background:#F5B700; color:var(--ink) !important; } .pill.outdoor { background:#2E8B57; color:#fff !important; }
.mapcard .attr { margin:0.45rem 0 0; font-size:0.72rem; color:#6F8499 !important; text-align:right; }
.lmap > .mapnote { position:absolute; top:52px; left:12px; right:12px; font-size:0.75rem; color:var(--ink) !important;
  background:rgba(255,255,255,.9); padding:0.3em 0.6em; border-radius:8px; }
.chart-title { color:var(--muted) !important; font-size:0.88rem; margin:0.6rem 0 0.2rem; }
.chart { width:100%; height:auto; overflow:visible; }
.chart .danger { stroke:#9FB3C6; stroke-dasharray:4 4; stroke-width:1.2; }
.chart .dlab, .chart .axis { fill:#6F8499; font-size:10px; font-family:Geist, system-ui, sans-serif; }
.chart .vlab { font-size:11px; font-weight:600; font-family:Geist, system-ui, sans-serif; }
.chartsec { margin-top:2.6rem; }
.chartsec h3 { font-size:1.6rem; letter-spacing:-0.02em; margin:0 0 0.3rem; color:var(--ink) !important; }
.chartsub { color:var(--muted) !important; max-width:70ch; margin:0 0 1rem; }
.nws { position:relative; }
.nwsmap { position:relative; width:100%; background-size:100% 100%; border-radius:18px; overflow:hidden;
  box-shadow: inset 0 0 0 1px var(--line); }
.nwssvg { position:absolute; inset:0; width:100%; height:100%; }
.cityl { font: 700 22px Geist, system-ui, sans-serif; fill:#0F2A43; paint-order:stroke; stroke:#fff; stroke-width:5px; }
.warrow { stroke:#0F2A43; stroke-width:2.5; stroke-linecap:round; } .whead { fill:#0F2A43; }
.nwstitle { position:absolute; z-index:3; top:12px; left:50%; transform:translateX(-50%); background:rgba(244,244,244,.94);
  border:1px solid #9AA5B1; border-radius:8px; padding:0.45rem 1rem; text-align:center; font-weight:600;
  color:var(--ink) !important; box-shadow:0 4px 14px rgba(15,42,67,.18); white-space:nowrap; }
.nwstitle span { font-weight:400; font-size:0.85rem; color:var(--muted) !important; }
.legend { position:absolute; left:14px; bottom:14px; list-style:none; margin:0; padding:0.7rem 0.9rem;
  background:rgba(244,244,244,.95); border:1px solid #9AA5B1; border-radius:8px; box-shadow:0 4px 14px rgba(15,42,67,.18);
  display:grid; grid-template-columns:repeat(2, auto); gap:0.15rem 0.9rem; font-size:0.74rem; }
.legend li { display:flex; align-items:center; gap:0.45rem; color:var(--ink) !important; }
.legend .lg-note { grid-column:1 / -1; color:var(--muted) !important; }
.sw { width:18px; height:12px; border-radius:3px; flex:none; border:1px solid rgba(0,0,0,.15); }
.arrow-sw { width:22px; text-align:center; font-weight:700; color:var(--ink); }
.chartsec .attr { margin:0.45rem 0 0; font-size:0.72rem; color:#6F8499 !important; }
.regline { margin-top:0.8rem; }
.aqi-top { display:flex; gap:1rem; align-items:center; }
.aqi-num { flex:none; width:76px; height:76px; border-radius:50%; display:grid; place-items:center; font-size:1.7rem;
  font-weight:700; box-shadow: inset 0 0 0 3px rgba(0,0,0,.08); }
.aqi-name { font-size:1.15rem; color:var(--ink) !important; } .aqi-advice { margin:0.2rem 0 0; color:var(--muted) !important; }
.aqi-scale { position:relative; display:flex; height:10px; border-radius:99px; overflow:visible; margin:1.1rem 0 0.3rem; }
.aqi-scale span:first-child { border-radius:99px 0 0 99px; } .aqi-scale span:last-of-type { border-radius:0 99px 99px 0; }
.aqi-scale i { position:absolute; top:-5px; width:4px; height:20px; background:var(--ink); border-radius:2px; transform:translateX(-50%); }
.aqi-scale-l { position:relative; height:1rem; font-size:0.7rem; }
.aqi-scale-l span { position:absolute; transform:translateX(-50%); color:#6F8499 !important; }
.aqi-scale-l span:first-child { transform:none; } .aqi-scale-l span:last-child { transform:translateX(-100%); }
.aqi-main { margin:0.9rem 0 0.4rem; color:var(--muted) !important; } .aqi-main b { color:var(--ink); }
.pollutants { list-style:none; margin:0; padding:0; display:grid; grid-template-columns:1fr 1fr; gap:0.4rem; }
.pollutants li { background:var(--cream); border-radius:10px; padding:0.45rem 0.6rem; font-size:0.82rem; color:var(--muted) !important; }
.pollutants span { display:block; color:var(--muted) !important; font-size:0.75rem; } .pollutants b { font-size:1.1rem; color:var(--ink); }
.rose-row { display:grid; grid-template-columns:2.2rem 1fr 2.4rem; gap:0.6rem; align-items:center; padding:0.22rem 0; }
.rose-row span { color:var(--muted) !important; font-size:0.88rem; }
.rtrack { height:12px; background:var(--cream); border-radius:99px; overflow:hidden; }
.rtrack span { display:block; height:100%; background:#9FB3C6; border-radius:99px; }
.rose-row.now .rtrack span { background:var(--orange); } .rose-row.now span.rd, .rose-row.now span.rv { color:var(--ink) !important; font-weight:700; }
.rv { text-align:right; font-variant-numeric:tabular-nums; }
.months { display:grid; grid-template-columns:repeat(12, 1fr); gap:0.35rem; height:160px; margin-top:0.5rem; }
.mcol { display:flex; flex-direction:column; align-items:center; gap:0.25rem; }
.mbars { flex:1; width:100%; display:flex; align-items:flex-end; justify-content:center; gap:2px; }
.mbars span { width:42%; border-radius:4px 4px 0 0; } .ph, .key.ph { background:#0F2A43; } .de, .key.de { background:#C77C00; }
.mcol i { font-style:normal; font-size:0.72rem; color:#6F8499; }
.key { display:inline-block; width:10px; height:10px; border-radius:2px; margin:0 0.25rem 0 0.6rem; vertical-align:middle; }
.scores, .how { margin-top:2.6rem; }
.scores h3, .how h3 { font-size:1.6rem; letter-spacing:-0.02em; margin:0 0 0.3rem; color:var(--ink) !important; }
.scores > p, .how p { color:var(--muted) !important; max-width:62ch; line-height:1.55; }
.srow { display:grid; grid-template-columns: 1.6fr 1fr 1fr 1fr; padding:1rem 1.2rem; border-radius:14px; margin-top:0.6rem;
  align-items:baseline; }
.srow.ours { background:var(--ink); } .srow.ours, .srow.ours span { color:var(--cream) !important; } .srow.ours b { color:var(--orange) !important; }
.srow.base { background:var(--paper); border:1px solid var(--line); } .srow.base span, .srow.base b { color:var(--ink) !important; }
.srow b { font-size:1.5rem; letter-spacing:-0.02em; } .sname { font-weight:600; }
.fine { color:var(--muted) !important; font-size:0.88rem; margin-top:2.4rem; border-top:1px solid var(--line); padding-top:1rem; }
/* Tablets and small laptops */
@media (max-width: 1080px) {
  .result { grid-template-columns: 1fr 1fr; } .panels { grid-template-columns: 1fr 1fr; }
  .meter { grid-template-columns: 1fr 2.8rem; } .meter .track { grid-column: 1 / -1; grid-row: 2; }
}
/* Phones and narrow tablets: everything in one column */
@media (max-width: 820px) {
  .result, .panels { grid-template-columns: 1fr; } .navlinks { display:none; }
  #picker { flex-direction: column !important; align-items: stretch !important; }
  #picker > * { width: 100% !important; min-width: 0 !important; }
  .srow { grid-template-columns: 1fr 1fr 1fr; } .sname { grid-column: 1 / -1; margin-bottom:0.3rem; }
  .sheet { position:static; margin-top:0.6rem; box-shadow:none; border:1px solid var(--line); }
  .legend { position:static; margin-top:0.6rem; grid-template-columns:1fr 1fr; }
  .nwstitle { position:static; transform:none; white-space:normal; font-size:0.85rem; margin-bottom:0.5rem; box-shadow:none; }
  .nws { display:flex; flex-direction:column; } .nws .nwstitle { order:-1; }
}
/* Small phones */
@media (max-width: 480px) {
  .call-side, .card { padding: 1.1rem; border-radius: 16px; } .board { padding: 1.1rem; box-shadow: 6px 6px 0 var(--orange); }
  .mapcard { padding: 0.8rem; } .chip { font-size:0.75rem; } .sheet-row { flex-wrap:wrap; } .srow { padding: 0.8rem 0.9rem; } .srow b { font-size: 1.25rem; }
  .nav { padding-bottom: 1rem; }
}
/* Big monitors */
@media (min-width: 1600px) {
  .result { grid-template-columns: 1fr 1.2fr; }
}
"""

HEAD = ('<link rel="preconnect" href="https://fonts.googleapis.com">'
        '<link href="https://fonts.googleapis.com/css2?family=Geist:wght@400;500;600;700'
        '&family=Instrument+Serif:ital@1&display=swap" rel="stylesheet">')

# Always show the light (cream) version, even if the computer is in dark mode
FORCE_LIGHT = "() => { document.body.classList.remove('dark'); }"

with gr.Blocks(title="Recess Ready: Philadelphia") as demo:
    gr.HTML(HERO)
    with gr.Row(elem_id="picker"):
        date_box = gr.Dropdown(test_dates, value=test_dates[0], label="Pick a school day from the 2025-26 test year", scale=3)
        go = gr.Button("Make the call", elem_id="go", scale=1)
    result = gr.HTML(recess_check(test_dates[0]))
    gr.HTML(scores_html())
    go.click(recess_check, inputs=date_box, outputs=result)
    date_box.change(recess_check, inputs=date_box, outputs=result)

demo.launch(css=CSS, head=HEAD, js=FORCE_LIGHT)
